In [ ]:
from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import alpha
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from itertools import combinations
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,ConfusionMatrixDisplay,classification_report,r2_score
from six import StringIO
from IPython.display import Image
from sklearn.tree import export_graphviz
import pydotplus
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2, f_regression
from sklearn.linear_model import LinearRegression


fish_path = glob("../INPUT_FISH/Fish.csv")
# fish_path
fish_data_ = pd.read_csv(fish_path[0])
# fish_data_.shape
# fish_data_.isna().sum()
# fish_data_.Species.value_counts()
# fish_data_[fish_data_.Weight <= 0]
fish_data_ = fish_data_.dropna()
fish_data_df = fish_data_.drop([40])
le = LabelEncoder()

fish_data_df["Species_encoded"] = le.fit_transform(fish_data_df["Species"])
#
# dummy__ = pd.get_dummies(fish_data_df['Species'])
# fish_data_df_dummy = pd.concat([fish_data_df, dummy__], axis=1)
# fish_data_df_dummy = fish_data_df_dummy.drop('Species', axis=1)
fish_data_w_species = fish_data_df.drop(columns=['Species'])



linear_model = LinearRegression()

### Vérification des relations entre les variables

In [ ]:
correl = fish_data_w_species.corr()
plt.rcParams["figure.figsize"] = (10,6)
sns.heatmap(correl, annot =True, alpha=0.9)
plt.title('Correlation Matrix')

In [ ]:
# correl = fish_data_df_dummy.corr()
# plt.rcParams["figure.figsize"] = (10,6)
# sns.heatmap(correl, annot =True, alpha=0.9)
# plt.title('Correlation Matrix')

### Séparation des données de Train et Test

In [ ]:
feats = ['Length1', 'Length2', 'Length3', 'Height', 'Width', 'Species_encoded']
targ = 'Weight'

X = fish_data_w_species[feats]
y = np.ravel(fish_data_w_species[targ])

# selector = SelectKBest(score_func=f_regression, k=6)
# X_new = selector.fit_transform(X, y)
# selector.get_feature_names_out()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('Taille train:', X_train.shape, 'Taille test:', X_test.shape)

sns.pairplot(fish_data_df, kind = 'scatter', hue = 'Species')
# sns.pairplot(fish_data_df_dummy, kind = 'scatter')

### Entaînement  du Model

In [ ]:
results = []
for k in range(1, len(feats)+1):
    for combo in combinations(feats, k):
        # print(f" 1 - Combo Combo : {combo}")
        combo = list(combo)
        # print(f" 2 - Combo Combo : {combo}")
        linear_model = LinearRegression()
        linear_model.fit(X_train[combo], y_train)
        y_pred = linear_model.predict(X_test[combo])
        r2 = r2_score(y_test, y_pred)
        results.append({'feats': combo, 'r2': r2})

res_df = pd.DataFrame(results)
res_df = res_df.sort_values('r2', ascending=False).reset_index(drop=True)

res_df

###

In [ ]:
plt.figure(figsize=(8,6))
plt.barh(range(min(20, len(res_df))), res_df['r2'].head(20)[::-1])
labels = ['+'.join(f) for f in res_df['feats'].head(20)][::-1]
plt.yticks(range(min(20, len(res_df))), labels)
plt.xlabel('R2 (test)')
plt.title('Top 20 combinaisons de features par R2 (Linear Regression)')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

In [ ]:
X_train_n = X_train[res_df.feats[0]]
X_test_n = X_test[res_df.feats[0]]

### Régression Linéaire et Plot

In [ ]:
linear_model.fit(X_train_n, y_train)
y_pred_lin = linear_model.predict(X_test_n)

print('R2 (LinearRegression) sur test:', r2_score(y_test, y_pred_lin))


plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_lin)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('Weight réel')
plt.ylabel('Weight prédit')
plt.title('LinearRegression: réel vs prédit')
plt.grid(True)
plt.show()


### Performance indicators

In [ ]:
print(f"Performance Indicators of my models : {res_df.r2[0]}")